# 9-8 網格射線掃描（Raycasting）與連線阻擋（APCS g596 專題）

- **單元編號**：9-8
- **學習目標**：
  1. 理解網格「射線掃描（Raycasting）」模型，掌握以 `while` 迴圈沿固定方向向量連續推進座標的機制。
  2. 掌握射線遭遇地圖邊界時的「邊界終止條件（Boundary Stop）」，杜絕無窮迴圈與索引越界。
  3. 掌握射線遭遇網格物體時的「障礙阻擋與目標命中（Obstacle Stop）」判定邏輯。
  4. 實作十字四方向雷射掃描器（Cross Laser Scanner），同時向四個外擴方向發射探測射線。
  5. 掌握兩柱之間的「網格區間連線與染色（Line Segment Marking）」，學會對直線路徑上的格子進行批量標記。
  6. 完整貫通 APCS 經典中級實作真題 **g596 骨牌遊戲**（舊版實作第 2 題），在無自訂函式架構下完成柱子增減、連線重塑與被阻擋判定的完整動態模擬。
- **適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
- **先備知識**：9-1 二維陣列座標概念、9-4 二維走訪與統計、9-7 方向向量與相鄰探測

---


### 9.8.1 沿單一方向連續推進模型（步進 while 迴圈）

在上一單元（9-7）中，我們學會了向相鄰的格子「跨出一步」。但在許多射擊遊戲、雷達掃描、棋盤視線（如西洋棋的車、皇后、主教）中，角色往往需要「**朝著某個方向一路看過去，直到世界的盡頭**」！

這種「沿著某個固定方向不斷向前延伸」的演算法模型，在電腦圖學與遊戲程式中被稱為「**射線掃描（Raycasting）**」。

#### 🚀 射線推進的物理機制：
想像從座標 $(r, c)$ 發射一道雷射光：
- 如果雷射光往「右」發射，方向向量為 $(\Delta r, \Delta c) = (0, 1)$。
- 第 1 步：推進到 $(r + 0, c + 1)$
- 第 2 步：推進到 $(r + 0, c + 2)$
- 第 3 步：推進到 $(r + 0, c + 3)$
- 依此類推……

因為我們事先**不知道雷射光到底能走多少步**（可能 2 步就出界，也可能走 10 步），所以射線掃描的標準核心結構不是 `for` 迴圈，而是 **`while` 迴圈**！
在每一次迴圈迭代中，只要前進條件依然滿足，我們就將當前座標加上向量偏移量：
```python
nr += dr
nc += dc
```
每加一次，射線就向前推進一格。這就是二維射線掃描的最基本物理引擎！


In [ ]:
# 範例 9.8.1：從 (2, 1) 開始，向右發射射線並印出所有踩過的座標軌跡

demo_R, demo_C = 5, 6  # 5 列 6 行的網格
demo_start_r, demo_start_c = 2, 1  # 發射起點

# 定義向右前進的方向向量
demo_dr, demo_dc = 0, 1

print(f"射線從 ({demo_start_r}, {demo_start_c}) 向右發射！")

# 射線前進座標：初始設在起點往右跨出的第一格
demo_crr_r = demo_start_r + demo_dr
demo_crr_c = demo_start_c + demo_dc

# 當射線行座標還在地圖範圍內 (curr_c < C) 時，持續向前推進
demo_s098 = 1
while demo_crr_c < demo_C:
    print(f"第 {demo_s098} 步抵達座標: ({demo_crr_r}, {demo_crr_c})")
    demo_crr_r += demo_dr
    demo_crr_c += demo_dc
    demo_s098 += 1

print("射線已抵達地圖最右側邊界！")

In [ ]:
# 填空 9.8.1：向下發射射線並印出軌跡
# 請將 ___ 替換為正確的向量或座標更新

R, C = 6, 6
start_r, start_c = 1, 3

# 定義向下的方向向量：列加 1，行不變
dr098 = ___
dc098 = 0

curr_r = start_r + dr098
curr_c = start_c + dc098

# 當列座標還在 R 範圍內時，持續推進
while curr_r < ___:
    print(f"向下推進到: ({curr_r}, {curr_c})")
    curr_r += ___
    curr_c += ___

print("射線抵達地圖最底部！")


In [ ]:
# 練習 9.8.1：單向射線點數累加器
# 題目說明：輸入兩個整數 R, C 代表地圖大小，接著輸入 R x C 整數地圖。
# 最後一行輸入發射起點座標 r, c 與方向代碼 D（0 代表向右，1 代表向下）。
# 請從 (r, c) 沿著指定方向發射射線（不含起點自身），將射線所穿透的所有格子數值加總，並輸出總和。
# 若起點已經在該方向的邊界上（無法前進），則輸出 0。

# 【公開測試資料 1】
# 3 4
# 1 2 3 4
# 5 6 7 8
# 9 1 2 3
# 1 0 0
# 輸出：21 (起點(1,0)=5，向右穿透 (1,1)=6, (1,2)=7, (1,3)=8，總和 6+7+8 = 21)

# 【公開測試資料 2】
# 3 3
# 10 20 30
# 40 50 60
# 70 80 90
# 0 1 1
# 輸出：130 (起點(0,1)=20，向下穿透 (1,1)=50, (2,1)=80，總和 50+80 = 130)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
gd098 = []
for _ in range(R):
    gd098.append(list(map(int, input().split())))

r, c, D = map(int, input().split())

if D == 0:
    dr, dc = 0, 1
else:
    dr, dc = 1, 0

crr_r = r + dr
crr_c = c + dc
tot_sc098 = 0

while 0 <= crr_r < R and 0 <= crr_c < C:
    tot_sc098 += gd098[crr_r][crr_c]
    crr_r += dr
    crr_c += dc

print(tot_sc098)


In [ ]:
# 挑戰 9.8.1：向左與向上雙向反向射線
# 題目說明：輸入 R, C 與 R x C 整數矩陣。
# 輸入起點座標 r, c。請分別向左（dr=0, dc=-1）與向上（dr=-1, dc=0）發射射線。
# 比較兩條射線所穿透的數值總和，輸出較大的總和值。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
gd53 = []
for _ in range(R):
    gd53.append(list(map(int, input().split())))

r, c = map(int, input().split())

# 向左
left_sum = 0
cr, cc = r, c - 1
while cc >= 0:
    left_sum += gd53[cr][cc]
    cc -= 1

# 向上
up_sum = 0
cr, cc = r - 1, c
while cr >= 0:
    up_sum += gd53[cr][cc]
    cr -= 1

print(max(left_sum, up_sum))


### 9.8.2 射線撞牆邊界終止條件（Boundary Stop）

在進行二維射線掃描時，初學者最常犯的重大致命錯誤就是——「**讓射線無限飛出地圖，導致無窮迴圈或 `IndexError` 崩潰**」！

想像一束光發射出去，若地圖外圍是黑洞或牆壁，光線一旦飛出世界邊界，探測就必須立刻停止。這稱為「**邊界終止條件（Boundary Stop）**」。

在一個 $R \times C$ 的矩陣中，射線每推進一步前或推進一步後，必須立刻接受檢查：
```python
if not (0 <= nr < R and 0 <= nc < C):
    # 撞牆出界了！立刻中斷！
    break
```
或者直接寫在 `while` 迴圈的條件開頭：
```python
while 0 <= nr < R and 0 <= nc < C:
    # 安全走訪當前格
    # 接著繼續往前一步
    nr += dr
    nc += dc
```

#### 💡 邊界停止的雙重保護意義：
1. **防止無窮迴圈**：如果射線沒有出界終止條件，`while True` 永遠不會停止，在 OJ 上會被裁判系統判定為 `TLE`（超過時間限制）。
2. **防止負索引幽靈讀取**：向上推進時若減到 `-1`，Python 不會報錯而是悄悄存取矩陣最後一列，導致射線「穿透地圖反彈到背面」，引發荒謬的計算錯誤。
牢記：**邊界防護是射線推進的第一道生命線**！


In [ ]:
# 範例 9.8.2：使用通用邊界防護 while 迴圈發射斜角射線（向右下發射）

demo_R, demo_C = 4, 5
demo_start_r, demo_start_c = 1, 1

# 向右下的方向向量：列加 1，行加 1
demo_dr, demo_dc = 1, 1

demo_cr098 = demo_start_r + demo_dr
demo_cc098 = demo_start_c + demo_dc

print(f"從 ({demo_start_r}, {demo_start_c}) 向右下發射射線：")
while 0 <= demo_cr098 < demo_R and 0 <= demo_cc098 < demo_C:
    print(f"  射線穿過有效座標: ({demo_cr098}, {demo_cc098})")
    demo_cr098 += demo_dr
    demo_cc098 += demo_dc

print("【撞牆終止】射線已抵達邊界外，安全退出迴圈！")

In [ ]:
# 填空 9.8.2：向上發射射線並以 break 機制做邊界終止
# 請將 ___ 替換為正確的終止條件

R, C = 5, 5
start_r, start_c = 3, 2

dr, dc = -1, 0  # 向上
crr_r098 = start_r + dr
crr_c098 = start_c + dc

while True:
    # 檢查是否超出邊界（上邊界為 crr_r098 < 0）
    if crr_r098 < 0 or crr_r098 >= ___ or crr_c098 < 0 or crr_c098 >= ___:
        print("射線已撞牆，終止探測！")
        ___  # 跳出迴圈
        
    print(f"射線通過: ({crr_r098}, {crr_c098})")
    crr_r098 += dr
    crr_c098 += dc


In [ ]:
# 練習 9.8.2：邊界阻斷射線長度計數器
# 題目說明：輸入地圖大小 R, C 與發射起點座標 r, c。
# 接著輸入前進方向向量 dr, dc（保證皆為 -1, 0 或 1 之一，且不同時為 0）。
# 請計算該射線從起點出發，在「撞到地圖邊界出界之前」，總共能在地圖內部跨出幾步（不包含起點）？

# 【公開測試資料 1】
# 5 5
# 2 2
# 0 1
# 輸出：2 (起點在 (2,2)，向右依序為 (2,3), (2,4)，抵達邊界共 2 步)

# 【公開測試資料 2】
# 4 6
# 0 3
# -1 0
# 輸出：0 (起點 (0,3) 向上一步 (-1,3) 即出界，可走 0 步)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
r, c = map(int, input().split())
dr, dc = map(int, input().split())

stps098 = 0
crr_r2 = r + dr
crr_c2 = c + dc

while 0 <= crr_r2 < R and 0 <= crr_c2 < C:
    stps098 += 1
    crr_r2 += dr
    crr_c2 += dc

print(stps098)


In [ ]:
# 挑戰 9.8.2：四邊界撞擊距離雷達
# 題目說明：輸入地圖尺寸 R, C 與中心座標 r, c (0 <= r < R, 0 <= c < C)。
# 請分別計算從該點出發，朝「上、下、左、右」四個方向各自走到邊界外的有效步數，
# 依序印出 4 個步數（以空白分隔）。
# 本題無公開測試資料，請自行測試角落與邊界位置。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
r, c = map(int, input().split())

up_steps = r
down_steps = R - 1 - r
left_steps = c
rght_stps = C - 1 - c

print(up_steps, down_steps, left_steps, rght_stps)


### 9.8.3 射線遇障礙物阻擋機制（Obstacle Stop）

除了地圖邊界的牆壁之外，射線在行進過程中更常見的情境是——「**被地圖上的障礙物或目標物擋住**」！

例如在視線判斷（Line of Sight）中，如果你和敵人間隔著一堵石牆，雷射就無法穿透過去；在骨牌遊戲（APCS g596）中，兩根木柱之間如果有其他木柱，連線就會被該木柱阻擋。

因此，射線掃描在推進時，具有**雙重中斷條件**：
1. **條件 A：撞牆出界**（`not (0 <= nr < R and 0 <= nc < C)`） $\rightarrow$ 終止，未命中任何物體。
2. **條件 B：撞見物體**（`grid[nr][nc] != 0` 或等於特定標記） $\rightarrow$ **命中物體，射線被阻擋**，記錄命中資訊後終止！

這種結構在程式碼中體現為一個典型的「探測迴圈」：
```python
hit_obstacle = False
hit_r, hit_c = -1, -1

while 0 <= curr_r < R and 0 <= curr_c < C:
    if grid[curr_r][curr_c] == 1:  # 假設 1 代表障礙物
        hit_obstacle = True
        hit_r, hit_c = curr_r, curr_c
        break  # 被障礙物擋住了！不再繼續往前射！
    curr_r += dr
    curr_c += dc
```
透過 `break` 與命中標記，我們能清楚區分射線究竟是「射到虛空中出界」，還是「精準擊中了前方障礙物」。


In [ ]:
# 範例 9.8.3：向右發射雷射，檢測是否擊中障礙物（1 代表障礙物，0 代表空地）

demo_gd098 = [
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0]
]

demo_R, demo_C = len(demo_gd098), len(demo_gd098[0])
demo_start_r, demo_start_c = 1, 0  # 從 (1, 0) 向右射

demo_dr, demo_dc = 0, 1
demo_crrr2 = demo_start_r + demo_dr
demo_crrc2 = demo_start_c + demo_dc

demo_hfnd = False
demo_hpos = None

while 0 <= demo_crrr2 < demo_R and 0 <= demo_crrc2 < demo_C:
    if demo_gd098[demo_crrr2][demo_crrc2] == 1:
        demo_hfnd = True
        demo_hpos = (demo_crrr2, demo_crrc2)
        break  # 被 (1, 2) 處的障礙物阻擋！
    demo_crrr2 += demo_dr
    demo_crrc2 += demo_dc

if demo_hfnd:
    print(f"射線在 {demo_hpos} 被障礙物擋住了！")
else:
    print("射線一路暢行無阻，直接飛出地圖邊界！")

In [ ]:
# 填空 9.8.3：向下探測第一個障礙物
# 請將 ___ 替換為正確的變數或中斷指令

brd098 = [
    [0, 0],
    [0, 0],
    [0, 1],
    [0, 0]
]

R098 = len(brd098)
C098 = len(brd098[0])

curr_r, curr_c = 0, 1  # 從 (0, 1) 向下發射
dr, dc = 1, 0

hit_row = -1

while 0 <= curr_r < R098 and 0 <= curr_c < C098:
    if brd098[curr_r][curr_c] == 1:
        hit_row = curr_r
        ___  # 發現障礙物，立刻終止前進！
    curr_r += ___
    curr_c += ___

print("障礙物所在的列索引：", hit_row)
# 應輸出 2


In [ ]:
# 練習 9.8.3：尋找最近障礙物距離
# 題目說明：輸入地圖大小 R, C，接著輸入 R x C 的地圖（0 代表平地，1 代表石頭）。
# 最後輸入起點座標 r, c 與方向向量 dr, dc。
# 請沿著該方向發射射線：
# 若有擊中石頭，請輸出從起點到該石頭的「步數距離」（步數）；
# 若一路無阻直到出界，請輸出 -1。

# 【公開測試資料 1】
# 3 5
# 0 0 0 0 0
# 0 0 0 1 0
# 0 0 0 0 0
# 1 1 0 1
# 輸出：2 (起點在(1,1)，向右發射，在 (1,3) 擊中石頭，距離為 2 步)

# 【公開測試資料 2】
# 3 3
# 0 1 0
# 0 0 0
# 0 0 0
# 2 0 -1 0
# 輸出：-1 (起點在(2,0)，向上發射，一路經過 (1,0), (0,0) 皆無石頭，出界，輸出 -1)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
gd54 = []
for _ in range(R):
    gd54.append(list(map(int, input().split())))

r, c = map(int, input().split())
dr, dc = map(int, input().split())

dist = 0
crr_r3 = r + dr
crr_c3 = c + dc
hit = False

while 0 <= crr_r3 < R and 0 <= crr_c3 < C:
    dist += 1
    if gd54[crr_r3][crr_c3] == 1:
        hit = True
        break
    crr_r3 += dr
    crr_c3 += dc

if hit:
    print(dist)
else:
    print(-1)


In [ ]:
# 挑戰 9.8.3：射線穿透力測試員
# 題目說明：輸入 R, C 與 R x C 地圖（0 代表空地，1 代表普通障礙物，2 代表鋼鐵牆）。
# 射線擁有「穿透 1 次普通障礙物」的能力：
# 當遇到第一個 1 時，將其擊碎並繼續向前；當遇到第二個 1 或任何一個 2 時，射線立即停止。
# 請輸出射線最終停止時所在的座標 (r, c)；若飛出地圖邊界，輸出 "Out of Bounds"。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
gd55 = []
for _ in range(R):
    gd55.append(list(map(int, input().split())))

r, c = map(int, input().split())
dr, dc = map(int, input().split())

brkn_cnt = 0
crr_r4 = r + dr
crr_c4 = c + dc
stop_pos = None

while 0 <= crr_r4 < R and 0 <= crr_c4 < C:
    cell_val = gd55[crr_r4][crr_c4]
    if cell_val == 2:
        stop_pos = (crr_r4, crr_c4)
        break
    elif cell_val == 1:
        if brkn_cnt == 0:
            brkn_cnt += 1
        else:
            stop_pos = (crr_r4, crr_c4)
            break
    crr_r4 += dr
    crr_c4 += dc

if stop_pos:
    print(stop_pos[0], stop_pos[1])
else:
    print("Out of Bounds")


### 9.8.4 十字四方向雷射掃描器（Cross Laser Scanner）

現在，讓我們把單一方向的射線，升級為全方位的防禦系統——「**十字四方向雷射掃描器**」！

當一個物體（如防禦塔、炸彈或骨牌柱子）被放置在座標 $(r, c)$ 時，它會同時朝「**上、下、左、右**」四個方向發射射線進行廣播探測。

這時，我們需要將「方向向量迴圈 `for d in range(4):`」與「射線推進迴圈 `while`」進行**巢狀組合**：
- 外層迴圈：依序切換 4 個方向（$d = 0, 1, 2, 3$），分別取出對應的 $(dr[d], dc[d])$。
- 內層迴圈：針對當前取出的方向，啟動 `while` 射線推進，直到出界或遇到障礙物。

#### 📊 巢狀射線結構骨架：
```python
dr = [-1, 1, 0, 0]
dc = [0, 0, -1, 1]

for d in range(4):
    curr_r = r + dr[d]
    curr_c = c + dc[d]
    while 0 <= curr_r < R and 0 <= curr_c < C:
        # 探測該方向的格子
        if grid[curr_r][curr_c] == 1:
            # 找到該方向的第一個目標！
            break
        curr_r += dr[d]
        curr_c += dc[d]
```
這種「外層枚舉方向、內層射線推進」的雙層架構，正是 APCS g596 骨牌遊戲中最核心的感知骨幹！


In [ ]:
# 範例 9.8.4：十字雷射掃描——尋找中心點四個方向上最近的障礙物

demo_gd11 = [
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 0, 1, 0]
]

demo_R, demo_C = 4, 4
demo_center_r, demo_center_c = 1, 1  # 中心發射點 (1, 1)

demo_dr098 = [-1, 1,  0, 0]
demo_dc098 = [ 0, 0, -1, 1]
demo_dn098 = ["上", "下", "左", "右"]

print(f"中心發射台位於 ({demo_center_r}, {demo_center_c})")

for demo_d in range(4):
    demo_curr_r = demo_center_r + demo_dr098[demo_d]
    demo_curr_c = demo_center_c + demo_dc098[demo_d]
    demo_found_target = None
    
    while 0 <= demo_curr_r < demo_R and 0 <= demo_curr_c < demo_C:
        if demo_gd11[demo_curr_r][demo_curr_c] == 1:
            demo_found_target = (demo_curr_r, demo_curr_c)
            break  # 找到最近目標，終止該方向
        demo_curr_r += demo_dr098[demo_d]
        demo_curr_c += demo_dc098[demo_d]
        
    if demo_found_target:
        print(f"方向 [{demo_dn098[demo_d]}] 最近目標在：{demo_found_target}")
    else:
        print(f"方向 [{demo_dn098[demo_d]}] 射線直通天際，無任何障礙！")

In [ ]:
# 填空 9.8.4：統計十字四方向總共能照射到幾格空地（直到遇障礙或出界）
# 請將 ___ 替換為正確的變數或向量

R, C = 5, 5
center_r, center_c = 2, 2

dr12 = [-1, 1, 0, 0]
dc12 = [0, 0, -1, 1]

lit_count = 0

for d in range(4):
    curr_r = center_r + dr12[d]
    curr_c = center_c + dc12[d]
    while 0 <= curr_r < R and 0 <= curr_c < C:
        lit_count += 1
        curr_r += dr12[___]
        curr_c += dc12[___]

print("十字四方向照射到的總空地格數：", lit_count)
# 4 個方向各走 2 格出界，總共 8 格


In [ ]:
# 練習 9.8.4：十字雷射滅怪器
# 題目說明：輸入地圖尺寸 R, C 與 R x C 地圖（0 代表平地，1 代表怪物）。
# 接著輸入雷射塔發射座標 r, c。
# 雷射塔會同時向上下左右發射高能射線，射線穿透力極強，會貫穿該方向「所有的怪物直到地圖邊緣」。
# 請統計這座雷射塔在一次十字齊射中，總共能消滅多少隻怪物？

# 【公開測試資料 1】
# 3 5
# 0 1 0 1 0
# 1 0 1 1 0
# 0 1 0 0 0
# 1 1
# 輸出：3
# (發射點 (1,1)。
#  向上(0,1)有 1 隻怪；
#  向下(2,1)有 1 隻怪；
#  向左(1,0)有 1 隻怪；
#  向右(1,2)是怪、(1,3)也是怪 -> 等等，向右有 2 隻怪，總共 1+1+1+2 = 5 隻？
#  我們檢查：(0,1)=1, (2,1)=1, (1,0)=1, (1,2)=1, (1,3)=1，總共消滅 5 隻！)

# 【公開測試資料 2】
# 3 3
# 0 0 0
# 0 0 0
# 0 0 0
# 1 1
# 輸出：0 (地圖上完全沒有怪物)

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
gd56 = []
for _ in range(R):
    gd56.append(list(map(int, input().split())))

r, c = map(int, input().split())

dr13 = [-1, 1, 0, 0]
dc13 = [0, 0, -1, 1]

mnst_klld = 0

for d in range(4):
    curr_r = r + dr13[d]
    curr_c = c + dc13[d]
    while 0 <= curr_r < R and 0 <= curr_c < C:
        if gd56[curr_r][curr_c] == 1:
            mnst_klld += 1
        curr_r += dr13[d]
        curr_c += dc13[d]

print(mnst_klld)


In [ ]:
# 挑戰 9.8.4：最佳雷射塔選址
# 題目說明：輸入地圖尺寸 R, C 與 R x C 地圖（1 代表怪物，0 代表平地）。
# 請走訪全圖所有「平地格子 (gd57[r][c] == 0)」，計算若在該格建立雷射塔，向四個方向齊射能消滅的最大怪物總數。
# 輸出該最大擊殺數。若全圖皆無平地或無怪物，輸出 0。
# 本題無公開測試資料，請自行測試。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
gd57 = []
for _ in range(R):
    gd57.append(list(map(int, input().split())))

dr14 = [-1, 1, 0, 0]
dc14 = [0, 0, -1, 1]

max_kills = 0

for r in range(R):
    for c in range(C):
        if gd57[r][c] == 0:
            kills = 0
            for d in range(4):
                curr_r = r + dr14[d]
                curr_c = c + dc14[d]
                while 0 <= curr_r < R and 0 <= curr_c < C:
                    if gd57[curr_r][curr_c] == 1:
                        kills += 1
                    curr_r += dr14[d]
                    curr_c += dc14[d]
            if kills > max_kills:
                max_kills = kills

print(max_kills)


### 9.8.5 雙柱連線與網格區間染色（Line Segment Marking）

在 **APCS g596 骨牌遊戲** 中，核心的幾何規則是：
當我們在網格中立起柱子時，若兩根柱子位在「同一直線」上（同行或同列），且「**兩柱之間沒有任何其他柱子阻隔**」，這兩根柱子之間就會拉起一條連線，將兩柱之間的所有格子全部「**染色（Marking）**」！

這在程式實作上包含了兩階段的動作：
1. **連線判定（Raycast Check）**：
   從柱子 A 出發向某個方向發射射線，如果在出界前遇到的「**第一個物體**」恰好是柱子 B，則判定 A 與 B 成功連線！
2. **區間染色（Interval Marking）**：
   一旦確認連線成功，我們需要將 A 到 B 之間的所有格子標記為連線狀態。
   例如從柱子 A `(r1, c1)` 到柱子 B `(r1, c2)`（水平連線），我們只需將行索引從 $\min(c1, c2) + 1$ 跑到 $\max(c1, c2) - 1$：
   ```python
   for c in range(min(c1, c2) + 1, max(c1, c2)):
       marked[r1][c] = True
   ```
同理，如果是垂直連線，則固定行索引，走訪兩柱之間的列索引區間。

特別注意：兩根柱子「中間不能有第三根柱子」，如果中間有第三根柱子 C，射線撞到 C 就停了，A 就絕對不能直接連到 B！這種「最近視線阻擋」的嚴謹性，正是網格模擬題中最常考的邏輯點。


In [ ]:
# 範例 9.8.5：在同一列上的兩根柱子之間拉線染色

# 建立 3x6 網格，全為 0
demo_R, demo_C = 3, 6
demo_gd12 = [[0] * demo_C for _ in range(demo_R)]

# 在 (1, 1) 與 (1, 4) 分別立上一根柱子 (標記為 9)
demo_p1098 = (1, 1)
demo_p2098 = (1, 4)
demo_gd12[demo_p1098[0]][demo_p1098[1]] = 9
demo_gd12[demo_p2098[0]][demo_p2098[1]] = 9

print("立柱後的地圖：")
for demo_row in demo_gd12:
    print(*demo_row)

# 執行兩柱之間的「區間連線染色」（標記為 1）
# 兩柱同行同列，此處在同一列 (r = 1)
demo_r = demo_p1098[0]
demo_cstrt = min(demo_p1098[1], demo_p2098[1]) + 1
demo_c_end = max(demo_p1098[1], demo_p2098[1])

for demo_c in range(demo_cstrt, demo_c_end):
    demo_gd12[demo_r][demo_c] = 1

print()
print("連線染色後的地圖（1 代表連線骨牌）：")
for demo_row in demo_gd12:
    print(*demo_row)

In [ ]:
# 填空 9.8.5：垂直兩柱區間連線染色
# 請將 ___ 替換為正確的索引範圍

R, C = 6, 3
brd5 = [[0] * C for _ in range(R)]

# 垂直立柱於 (1, 1) 與 (4, 1)
r1, c1 = 1, 1
r2, c2 = 4, 1
brd5[r1][c1] = 9
brd5[r2][c2] = 9

# 兩柱在同一行 (c = 1)，在 r1 與 r2 之間填入連線標記 1
col = c1
for r in range(min(r1, r2) + 1, ___):
    brd5[r][___] = 1

print("垂直連線後的中間兩格：")
print("座標 (2, 1) =", brd5[2][1])
print("座標 (3, 1) =", brd5[3][1])


In [ ]:
# 練習 9.8.5：水平連線安全染色員
# 題目說明：輸入地圖大小 R, C，接著輸入兩根柱子的座標 r1, c1 與 r2, c2。
# 題目保證兩柱在同一列（r1 == r2 且 c1 != c2）。
# 請建立一個 R x C 全為 0 的矩陣，將兩柱位置設為 9，並將兩柱之間的所有格子填入 1，最後依序輸出整個矩陣。

# 【公開測試資料 1】
# 2 5
# 0 1 0 4
# 輸出：
# 0 9 1 1 9
# 0 0 0 0 0

# 【公開測試資料 2】
# 3 4
# 2 3 2 0
# 輸出：
# 0 0 0 0
# 0 0 0 0
# 9 1 1 9

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
r1, c1, r2, c2 = map(int, input().split())

gd58 = [[0] * C for _ in range(R)]
gd58[r1][c1] = 9
gd58[r2][c2] = 9

c_start = min(c1, c2) + 1
c_end = max(c1, c2)

for c in range(c_start, c_end):
    gd58[r1][c] = 1

for row in gd58:
    print(*row)


In [ ]:
# 挑戰 9.8.5：第三柱阻擋判定
# 題目說明：輸入 R, C 與兩柱座標 r1, c1 與 r2, c2（同行或同列）。
# 接著輸入第三柱座標 r3, c3。
# 檢查第三柱是否剛好卡在第一柱與第二柱的「正中間線上」？
# 若被阻擋輸出 "Blocked"，若未被阻擋輸出 "Connected"。
# 本題無公開測試資料，請自行測試不同位置。

# 請在此處撰寫你的程式碼：
R, C = map(int, input().split())
r1, c1, r2, c2 = map(int, input().split())
r3, c3 = map(int, input().split())

is_blocked = False

if r1 == r2 == r3:
    if min(c1, c2) < c3 < max(c1, c2):
        is_blocked = True
elif c1 == c2 == c3:
    if min(r1, r2) < r3 < max(r1, r2):
        is_blocked = True

if is_blocked:
    print("Blocked")
else:
    print("Connected")


### 9.8.6 APCS g596 骨牌遊戲：柱子拔除與連線阻擋動態模擬（APCS 實戰原型）

現在，我們來到本單元的壓軸真題——**APCS 實作真題 g596 骨牌遊戲**！

#### 📜 題意規則全景解析：
在一個 $R \times C$ 的網格中，初始時全為空地。
接著依序發生 $B$ 次操作，每次操作包含三筆整數：`r, c, t`
- `r, c` 代表目標座標。
- `t = 0` 代表「**放置柱子**」：在 `(r, c)` 立起一根柱子。
- `t = 1` 代表「**拔除柱子**」：將 `(r, c)` 的柱子拔掉，柱子消失。

每次操作後，柱子之間會建立連線（排上骨牌）：
1. 每一根現存的柱子，向「**上、下、左、右**」四個方向發射射線。
2. 若在某個方向遇到的**第一個物體**也是一根現存柱子，且兩柱之間的距離小於或等於限制 $K$（即兩柱距離 $|r1-r2| + |c1-c2| \le K$），則這兩柱成功連線！
3. 連線兩柱之間的所有格子都會排上骨牌（被標記為連線）。
4. 當柱子被拔除時，原本由它建立的連線會隨之瓦解（除非其他柱子之間也剛好穿過該處）。
5. 題目要求：在所有操作完成後，統計**全地圖中當前有骨牌（連線）或有柱子的「總格數」**，並輸出最終的網格總覆蓋數！

#### 🛠️ 高效重繪策略（Redraw Architecture）：
在競賽中，處理「動態拔除連線」最不易出錯的策略是「**全量重繪法（Full Redraw）**」：
- 我們用一張二維陣列維護「哪些格子當前有柱子」（例如 `has_post[r][c] = True / False`）。
- 每次操作：如果是 0 就設 `has_post = True`，如果是 1 就設 `has_post = False`。
- 在所有指令執行完畢後（或每次結算時），建立一張乾淨的 `covered` 矩陣。
- 走訪所有現存的柱子，向四個方向進行射線掃描（`while` 推進）：
  - 遇到地圖邊界或距離超過 $K$，停止；
  - 若遇到第一根柱子，則將兩柱之間的所有格子在 `covered` 上標記為 `True`！
- 最後統計 `covered` 中為 `True` 的總格數與柱子總數，即為完美解答！


In [ ]:
# 範例 9.8.6：APCS g596 核心重繪與四方向射線連線模擬

demo_R, demo_C, demo_K = 4, 5, 4  # 4x5 地圖，最大連線跨距 K = 4

# 當前地圖上有三根柱子：(1, 1), (1, 4), (3, 1)
demo_posts = [
    [False] * demo_C for _ in range(demo_R)
]
demo_posts[1][1] = True
demo_posts[1][4] = True
demo_posts[3][1] = True

# covered 陣列記錄覆蓋（含柱子與骨牌連線）
demo_cvrd = [[False] * demo_C for _ in range(demo_R)]

# 柱子本身一定被覆蓋
for demo_r in range(demo_R):
    for demo_c in range(demo_C):
        if demo_posts[demo_r][demo_c]:
            demo_cvrd[demo_r][demo_c] = True

demo_dr5 = [-1, 1, 0, 0]
demo_dc5 = [0, 0, -1, 1]

# 走訪每一根現存柱子，向四方向發射射線尋找連線
for demo_r in range(demo_R):
    for demo_c in range(demo_C):
        if not demo_posts[demo_r][demo_c]:
            continue
            
        for demo_d in range(4):
            demo_curr_r = demo_r + demo_dr5[demo_d]
            demo_curr_c = demo_c + demo_dc5[demo_d]
            demo_steps = 1
            
            while 0 <= demo_curr_r < demo_R and 0 <= demo_curr_c < demo_C and demo_steps <= demo_K:
                if demo_posts[demo_curr_r][demo_curr_c]:
                    # 找到了最近的另一根柱子！在 r 到 curr_r 之間全部覆蓋
                    demo_mark_r = demo_r + demo_dr5[demo_d]
                    demo_mark_c = demo_c + demo_dc5[demo_d]
                    while (demo_mark_r, demo_mark_c) != (demo_curr_r, demo_curr_c):
                        demo_cvrd[demo_mark_r][demo_mark_c] = True
                        demo_mark_r += demo_dr5[demo_d]
                        demo_mark_c += demo_dc5[demo_d]
                    break  # 射線被第一根柱子擋住，終止推進
                demo_curr_r += demo_dr5[demo_d]
                demo_curr_c += demo_dc5[demo_d]
                demo_steps += 1

demo_tcvrd = 0
for demo_r in range(demo_R):
    for demo_c in range(demo_C):
        if demo_cvrd[demo_r][demo_c]:
            demo_tcvrd += 1

print("覆蓋狀態地圖（1 代表覆蓋）：")
for demo_r in range(demo_R):
    demo_row_str = ["1" if demo_cvrd[demo_r][demo_c] else "0" for demo_c in range(demo_C)]
    print(*demo_row_str)

print("最終被骨牌或柱子覆蓋的總格數：", demo_tcvrd)

In [ ]:
# 填空 9.8.6：補齊 g596 射線搜尋另一根柱子的終止條件
# 請將 ___ 替換為正確的變數或判斷

# 假設當前正從 (r, c) 朝方向 d 推進
# posts[r][c] 為 True 代表此處有柱子
while 0 <= curr_r < R and 0 <= curr_c < C and steps <= ___:
    # 如果遇到另一根柱子
    if posts[curr_r][curr_c]:
        # 標記兩柱之間的所有格子
        mark_r = r + dr[d]
        mark_c = c + dc[d]
        while (mark_r, mark_c) != (curr_r, curr_c):
            covered[mark_r][mark_c] = True
            mark_r += dr[d]
            mark_c += dc[d]
        ___  # 射線已被柱子阻擋，跳出 while 迴圈！
    curr_r += dr[d]
    curr_c += dc[d]
    steps += 1


In [ ]:
# 練習 9.8.6：APCS g596 骨牌遊戲實戰（單純立柱版）
# 題目說明：輸入四個整數 R, C, K, B 代表地圖列數、行數、最大連線跨距與操作次數。
# 接著有 B 行，每行三個整數 r, c, t（此題 t 皆為 0，代表立起柱子）。
# 若同一直線上兩柱之間無其他柱子且步數 <= K，則兩柱間連線。
# 請輸出所有操作完成後，地圖上被柱子與骨牌連線覆蓋的「總格數」。

# 【公開測試資料 1】
# 4 5 4 3
# 1 1 0
# 1 4 0
# 3 1 0
# 輸出：6
# (柱子 3 根：(1,1), (1,4), (3,1)。
#  (1,1)與(1,4)距離 3 <= 4，中間 (1,2),(1,3) 連線；
#  (1,1)與(3,1)距離 2 <= 4，中間 (2,1) 連線；
#  總覆蓋格為 3 柱 + 3 連線 = 6 格)

# 【公開測試資料 2】
# 3 3 2 2
# 0 0 0
# 2 2 0
# 輸出：2 (兩柱在對角線上，無法連線，僅覆蓋兩柱自身共 2 格)

# 請在此處撰寫你的程式碼：


In [ ]:
# 挑戰 9.8.6：APCS g596 完整版（含動態拔除 t=1 與最大覆蓋歷史紀錄）
# 題目說明：輸入 R, C, K, B，且 t 包含 0（立柱）與 1（拔柱）。
# 每次操作後，除了維護地圖覆蓋外，同時追蹤「在整個過程中，任何時刻所出現過的最大覆蓋格數」。
# 最後輸出兩行：
# 第一行：所有操作結束時的「最終覆蓋格數」
# 第二行：過程中的「歷史最大覆蓋格數」
# 本題無公開測試資料，請自行測試包含拔柱操作的測資。

# 請在此處撰寫你的程式碼：
R, C, K, B = map(int, input().split())
posts = [[False] * C for _ in range(R)]
dr15 = [-1, 1, 0, 0]
dc15 = [0, 0, -1, 1]

max_hstr = 0
fnl_cvrd = 0

for _ in range(B):
    pr, pc, t = map(int, input().split())
    if t == 0:
        posts[pr][pc] = True
    else:
        posts[pr][pc] = False
        
    covered = [[False] * C for _ in range(R)]
    for r in range(R):
        for c in range(C):
            if posts[r][c]:
                covered[r][c] = True
                
    for r in range(R):
        for c in range(C):
            if not posts[r][c]:
                continue
            for d in range(4):
                curr_r = r + dr15[d]
                curr_c = c + dc15[d]
                steps = 1
                while 0 <= curr_r < R and 0 <= curr_c < C and steps <= K:
                    if posts[curr_r][curr_c]:
                        mark_r = r + dr15[d]
                        mark_c = c + dc15[d]
                        while (mark_r, mark_c) != (curr_r, curr_c):
                            covered[mark_r][mark_c] = True
                            mark_r += dr15[d]
                            mark_c += dc15[d]
                        break
                    curr_r += dr15[d]
                    curr_c += dc15[d]
                    steps += 1
                    
    current_cnt = 0
    for r in range(R):
        for c in range(C):
            if covered[r][c]:
                current_cnt += 1
    if current_cnt > max_hstr:
        max_hstr = current_cnt
    fnl_cvrd = current_cnt

print(fnl_cvrd)
print(max_hstr)


---
### 🧪【自主學習成效驗收】單元 9-8 一鍵自動評分與身分認證

恭喜你完成了單元 9-8 的所有探索與挑戰！🎉

剛才在各題中建立的數值型態變數、運算式與語法結構，現在都乖乖躺在記憶體置物櫃裡囉。  
點擊下方的 **▶ 播放鍵**，系統將跳出輸入框讓您登記**姓名**與**學號座號**（分開獨立欄位，亦可直接按 Enter 採用自學預設值），並自動啟動 Google 官方帳號身分認證（記錄真實 Email 與 Google 顯示姓名），產出**學習成果成績單**並同步登錄後台試算表！

> 💡 **貼心小叮嚀**：如果評分報告顯示某題未通過，別灰心！只要回到上方對應題目的儲存格修正程式碼，**重新點擊該題播放鍵 ▶ 執行**，再回來重新執行此評分儲存格就可以刷新成績與紀錄囉！

> 🆘 **終極救援方案**：如果重新執行個別題目後，成績還是怪怪的（例如明明寫對了卻還是顯示 ❌），最保險的做法是**整本全部重跑**，一鍵重置記憶體、告別幽靈變數！**Google Colab** 請點上方選單「**執行階段 ➔ 重新啟動工作階段並全部執行**」（快捷鍵 `Ctrl + F9`）；**VS Code** 請於上方工具列依序點選「**Clear All Outputs ➔ Restart ➔ Run All**」。

In [ ]:
#@title 🧪 點擊此處取得單元 9-8 學習成效自動評分報告 { display-mode: "form" }
# ------------------------------------------------------------------------------
# 官方智慧全環境評分載入器（支援：本地開發庫優先 ➔ 公開正式庫雙重備援）
# ------------------------------------------------------------------------------
import os
import sys
import json
import logging
import urllib.request

# 靜音內部 httplib2 警告
logging.getLogger("google_auth_httplib2").setLevel(logging.ERROR)

# 1. 互動式輸入學員姓名與學號（分開獨立欄位，按 Enter 採用預設值）
print("👤【學員身分登記】（自學測試可直接按 Enter 鍵使用預設值）：")
student_name = input("請輸入您的姓名 [預設: 自學冒險者]：").strip() or "自學冒險者"
student_id = input("請輸入學號或座號 [預設: 無座號]：").strip() or "無座號"

# 2. 自動嘗試獲取 Google 帳號官方身分認證（真實 Email 與 Google 暱稱）
google_email = None
google_name = None
try:
    from google.colab import auth
    print("🔐 正在連結 Google 帳號進行身分認證（若跳出授權視窗，請點選允許）...")
    auth.authenticate_user()

    # 方法 1: 使用 Google Drive API v3 (about:get) 抓取真實 displayName 與 emailAddress
    # Colab ADC 預設包含 drive 權限，能 100% 精準獲取 Google 帳號中文顯示名稱（如：葉明昌）
    try:
        import google.auth
        from googleapiclient.discovery import build
        creds, _ = google.auth.default()
        drive_service = build('drive', 'v3', credentials=creds, cache_discovery=False)
        about = drive_service.about().get(fields="user(displayName,emailAddress)").execute()
        u_obj = about.get("user", {})
        google_name = u_obj.get("displayName")
        google_email = u_obj.get("emailAddress")
    except Exception:
        pass

    # 方法 2: 若 Drive API 失敗，嘗試透過 OAuth2 userinfo / tokeninfo 備援
    if not google_email or not google_name:
        try:
            import google.auth
            from google.auth.transport.requests import Request
            credentials, _ = google.auth.default()
            credentials.refresh(Request())
            token = credentials.token

            for ep in [
                f"https://www.googleapis.com/oauth2/v3/tokeninfo?access_token={token}",
                "https://www.googleapis.com/oauth2/v3/userinfo",
                "https://www.googleapis.com/oauth2/v2/userinfo"
            ]:
                try:
                    req = urllib.request.Request(
                        ep,
                        headers={"Authorization": f"Bearer {token}"}
                    )
                    with urllib.request.urlopen(req, timeout=5) as resp:
                        u_data = json.loads(resp.read().decode('utf-8'))
                        if not google_email:
                            google_email = u_data.get("email") or u_data.get("emailAddress")
                        if not google_name:
                            google_name = (
                                u_data.get("name")
                                or u_data.get("given_name")
                                or u_data.get("displayName")
                            )
                        if google_email and google_name:
                            break
                except Exception:
                    continue
        except Exception:
            pass

    # 方法 3: 若只有 email 但 Google API 沒給 name，以 email 前綴作為暱稱回退
    if google_email and not google_name:
        google_name = google_email.split("@")[0]

    if google_email:
        print(f"✅ 身分認證成功！已綁定 Google 帳號：{google_name} ({google_email})")
except Exception:
    # 非 Colab 環境、本機離線測試或略過授權
    pass

# 3. 身分資料獨立注入環境變數（學號與姓名完全分開）
os.environ["STUDENT_NAME"] = student_name
os.environ["STUDENT_ID"] = student_id
if google_email:
    os.environ["GOOGLE_EMAIL"] = google_email
if google_name:
    os.environ["GOOGLE_NAME"] = google_name

# 4. 評分核心腳本載入（本機優先 ➔ 遠端記憶體載入，絕不產生實體暫存檔）
# 找不到來源就換下一條。內容已拿到但執行失敗時，停止改試其他來源。
# 評分腳本放在獨立命名空間執行，成功後也不把名稱寫回筆記本。答案讀執行前的學生存放格快照。
_loaded = False
_load_error = None

# 4.1 優先嘗試讀取開發庫/本機檔案（本機離線開發測試零延遲）
_local_paths = [
    "web/graders/grade_9_8.py",
    "../graders/grade_9_8.py",
    "../../graders/grade_9_8.py",
    "../web/graders/grade_9_8.py",
    "../../web/graders/grade_9_8.py"
]
for _lp in _local_paths:
    if not os.path.exists(_lp):
        continue
    try:
        with open(_lp, "r", encoding="utf-8") as _f:
            _code = _f.read()
        _grader_ns = {"__builtins__": __builtins__, "__name__": "apcs_grader", "__STUDENT_ENV__": dict(globals())}
        try:
            exec(_code, _grader_ns)
        finally:
            del _grader_ns
        os.environ["GRADER_SOURCE"] = _lp
        _loaded = True
        break
    except Exception as _exc:
        _load_error = _exc
        break

# 4.2 若在雲端環境（Colab），自動透過公開正式庫雙重備援以純記憶體載入
if not _loaded and _load_error is None:
    _grader_urls = [
        "https://raw.githubusercontent.com/johnnyy-lab/APCS1to3/main/graders/grade_9_8.py",
        "https://johnnyy-lab.github.io/APCS1to3/graders/grade_9_8.py",
        "https://cdn.jsdelivr.net/gh/johnnyy-lab/APCS1to3@main/graders/grade_9_8.py"
    ]
    for _url in _grader_urls:
        try:
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(_req, timeout=10) as _resp:
                _code = _resp.read().decode("utf-8")
        except Exception:
            continue
        try:
            _grader_ns = {"__builtins__": __builtins__, "__name__": "apcs_grader", "__STUDENT_ENV__": dict(globals())}
            try:
                exec(_code, _grader_ns)
            finally:
                del _grader_ns
            os.environ["GRADER_SOURCE"] = _url
            _loaded = True
            break
        except Exception as _exc:
            _load_error = _exc
            break

if not _loaded:
    if _load_error is not None:
        print("❌ 評分程式載入失敗，請稍後重新執行。若仍失敗，請把這個畫面交給老師。")
        if os.environ.get("GRADER_DEBUG", "").strip().lower() in ("1", "true", "yes", "on"):
            _reason = str(_load_error).strip().splitlines()
            _line = _reason[0][:180] if _reason else ""
            if _line:
                print("（開發診斷：" + type(_load_error).__name__ + "：" + _line + "）")
            else:
                print("（開發診斷：" + type(_load_error).__name__ + "）")
    else:
        print("❌ 評分系統連線異常，請確認網路連線狀態（或稍後重新執行）")


## 本單元重點回顧與核心心法

在單元 9-8 中，我們將二維陣列的探測能力從「相鄰單步」擴展到了「無限遠射線」，並征服了 APCS g596 骨牌遊戲：

1. **射線掃描（Raycasting）推進模型**：
   - 使用 `while` 迴圈搭配固定向量 `(dr, dc)` 連續更新座標 `curr_r += dr; curr_c += dc`。
2. **撞牆邊界終止（Boundary Stop）**：
   - 守護條件：`0 <= curr_r < R and 0 <= curr_c < C`，徹底杜絕無窮迴圈與負索引穿透。
3. **障礙阻擋與命中判定（Obstacle Stop）**：
   - 雙重終止條件：先出界終止，或中途遇到物體以 `break` 終止並記錄命中座標。
4. **十字四方向雷射掃描器**：
   - 外層 `for d in range(4):` 切換方向，內層 `while` 射線推進，構築全方位視線感知雷達。
5. **雙柱區間連線染色**：
   - 射線找到第一根柱子後，以 `while` 或區間走訪對兩柱之間的空白格子批量蓋章覆蓋。
6. **APCS g596 重繪架構**：
   - 面對動態立柱與拔柱，以 `posts` 矩陣維護現狀，每次結算以全量射線掃描重繪 `covered` 覆蓋矩陣，穩健 AC！
